# Runbook chạy dự án trên Google Colab

Notebook này **chỉ để chạy trên Colab**. Nó clone repo, sinh artifacts, sao lưu Drive và mở tunnel.

## Làm 3 việc này TRƯỚC khi Run all

1. **Runtime → Change runtime type → T4 GPU → Save.** `config.py` đọc `DEVICE` lúc import, nên bật GPU sau khi đã chạy thì phải Restart runtime.
2. **Sửa dòng `REPO` ở cell dưới** thành địa chỉ repo của bạn.
3. Nếu repo là **private**, `git clone` sẽ hỏi user/token — dùng Personal Access Token làm mật khẩu.

## Về thời gian

| Bước | Thời gian |
|---|---|
| Cài thư viện | 2–4 phút |
| Sinh artifacts (`all`) | ~15–25 phút trên T4 |
| Build React + nạp 4 mô hình | 3–6 phút (lần đầu tải Qwen ~3 GB) |

> ⚠️ Phiên Colab ngắt là link tunnel chết. Làm demo và **chụp ảnh màn hình ngay trong phiên đó**.
> Đừng để tab idle quá ~90 phút, Colab sẽ tự ngắt.


## 1. Kiểm tra GPU — chạy cell này trước tiên

Nếu in ra `cpu` thì bạn chưa bật T4. Quay lại Runtime → Change runtime type.

In [1]:
!nvidia-smi -L || echo 'CHUA BAT GPU - Runtime > Change runtime type > T4 GPU'
!python --version

GPU 0: Tesla T4 (UUID: GPU-1497689c-b236-ca6e-7563-65f34431a95d)
Python 3.13.15


## 2. Clone repo

**Sửa dòng `REPO` bên dưới.** Chạy lại cell này lần hai sẽ tự `git pull` thay vì clone đè.

In [2]:
import os, subprocess

REPO = "https://github.com/MMMAlonea04/AI_Web_Apps_Streamlit_React.git"
DEST = "/content/ai_web_apps"

def run(*cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(list(cmd), check=True)

if os.path.isdir(os.path.join(DEST, ".git")):
    run("git", "-C", DEST, "pull")
else:
    run("git", "clone", REPO, DEST)

os.chdir(DEST)          # Colab dung chung mot tien trinh, cwd giu cho cac cell sau
print("thu muc:", os.getcwd())
run("git", "log", "--oneline", "-1")
print(os.listdir())

$ git clone https://github.com/MMMAlonea04/AI_Web_Apps_Streamlit_React.git /content/ai_web_apps
thu muc: /content/ai_web_apps
$ git log --oneline -1
['core', '.gitignore', 'requirements-colab.txt', 'data', 'streamlit_app.py', 'api', 'web', 'Dockerfile', 'README.md', '.git', 'requirements-dev.txt', 'notebooks', '.dockerignore', 'config.py', 'AI_Web_Apps_Streamlit_React.ipynb', '.gitattributes', '.github', 'tests', 'requirements-ci.txt', 'scripts', 'MODEL_CARD.md', 'requirements.txt', 'requirements-streamlit.txt', 'pyproject.toml']


## 3. Cài thư viện

Dùng `requirements-colab.txt` — **không** cài lại torch, vì Colab đã có bản khớp CUDA của nó.
Cài đè `requirements.txt` ở đây dễ làm hỏng CUDA và tốn vài phút tải vô ích.

In [3]:
!pip install -q -r requirements-colab.txt
!python -c "import config, torch; print('DEVICE =', config.DEVICE, '| GPU =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'khong co'); print('LLM   =', config.LLM_MODEL)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.8 MB/s eta 0:00:00
DEVICE = cuda | GPU = Tesla T4
LLM   = Qwen/Qwen2.5-1.5B-Instruct


## 4. Sinh artifacts — bước lâu nhất (~15–25 phút)

5 stage chạy theo thứ tự cố định: `data · classifier · detector · retrieval · rag`.

**Nếu lỗi ở một stage**, sửa xong chạy lại riêng stage đó, không cần chạy lại từ đầu:

```python
!python scripts/build_artifacts.py rag
```

In [4]:
!python scripts/build_artifacts.py all

Thư mục dự án: /content/ai_web_apps
Thiết bị: cuda · stage: ['data', 'classifier', 'detector', 'retrieval', 'rag']

=== data ===
↓ https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz
  TF Flowers: {'daisy': 633, 'dandelion': 898, 'roses': 641, 'sunflowers': 699, 'tulips': 799} | tổng: 3670
↓ https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip
  COCO128: 128 ảnh
  Kích thước ảnh (300 mẫu): rộng 159–500 px, cao 200–428 px
  hình: artifacts/figures/01_du_lieu.png
=== data xong sau 17s ===

=== classifier ===
  thiết bị=cuda · epochs=5 · batch=64 · train=2936 val=367 test=367
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 148MB/s]
  epoch 1/5 · train loss 0.855 acc 0.781 · val loss 0.933 acc 0.790 · 19s
  epoch 2/5 · train loss 0.645 acc 0.893 · val loss 0.907 acc 0.728 · 19s
  epoch 3/5 · train loss 0.554 acc 0.935 

## 5. Sao lưu NGAY — `/content` mất khi hết phiên

Cell trên chạy xong thì làm việc này trước khi làm bất cứ thứ gì khác.
Bỏ qua là mất 20 phút và phải train lại.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ai_web_apps && cp -r artifacts data/gallery /content/drive/MyDrive/ai_web_apps/
!du -sh /content/drive/MyDrive/ai_web_apps/*

Mounted at /content/drive
58M	/content/drive/MyDrive/ai_web_apps/artifacts
38M	/content/drive/MyDrive/ai_web_apps/gallery


## 6. Xem số đo trước khi chạy giao diện

Đây là **số thật để điền vào `MODEL_CARD.md`**. Đừng tự ước lượng.

In [6]:
import json, pathlib

for p in sorted(pathlib.Path('artifacts').rglob('*metrics*.json')):
    d = json.loads(p.read_text(encoding='utf-8'))
    keep = {k: v for k, v in d.items() if k != 'history' and not isinstance(v, dict)}
    print(f'--- {p} ---')
    print(json.dumps(keep, indent=1, ensure_ascii=False))
    if 'text_to_image_precision@10' in d:
        print('  text->image P@10:', d['text_to_image_precision@10'])

--- artifacts/classifier/metrics.json ---
{
 "test_accuracy": 0.9509536784741145,
 "test_macro_f1": 0.9508784565931061,
 "epochs": 5,
 "batch_size": 64,
 "device": "cuda",
 "model": "resnet18-imagenet-finetune"
}
--- artifacts/detector/metrics.json ---
{
 "mAP50": 0.669649061510062,
 "mAP50_95": 0.5024436941561841,
 "dataset": "coco128",
 "model": "yolo11n",
 "device": "cuda"
}
--- artifacts/rag_metrics.json ---
{
 "hit@1": 1.0,
 "hit@3": 1.0,
 "n_questions": 10,
 "embed_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
 "llm": "Qwen/Qwen2.5-1.5B-Instruct"
}
--- artifacts/retrieval/metrics.json ---
{
 "image_to_image_precision@5": 0.8759999999999999,
 "gallery_size": 628,
 "model": "openai/clip-vit-base-patch32",
 "device": "cuda"
}
  text->image P@10: {'daisy': 1.0, 'dandelion': 1.0, 'roses': 1.0, 'sunflowers': 1.0, 'tulips': 1.0}


## 7. Chạy cả hai giao diện + link công khai

Bước này build React, khởi động FastAPI (nạp cả 4 mô hình) và Streamlit, rồi mở Cloudflare Tunnel.
`serve.py` tự cài Node 22 nếu bản Node của Colab quá cũ cho Vite 8.

Mở link in ra trên điện thoại để kiểm tra, rồi **chụp ảnh màn hình** làm minh chứng cho báo cáo.

In [ ]:
!python scripts/serve.py all

node: 20.19
$ /tools/node/bin/npm install --no-audit --no-fund --loglevel=error
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 21 packages in 2s
⠋npm notice
npm notice New major version of npm available! 10.8.2 -> 12.1.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.1.0
npm notice To update run: npm install -g npm@12.1.0
npm notice
⠋$ /tools/node/bin/npm run build

> ai-web-apps@1.0.0 build
> vite build

vite v8.3.1 building client environment for production...
✓ 22 modules transformed.
computing gzip size...
dist/index.html                   0.39 kB │ gzip:  0.26 kB
dist/assets/index-DKc65XFl.css    2.11 kB │ gzip:  0.89 kB
dist/assets/index-Dj9fDPSX.js   228.26 kB │ gzip: 71.67 kB

✓ built in 178ms
⠙React build: /content/ai_web_apps/web/dist
API sẵn sàng sau 42s: {'status': 'ok', 'device': 'cuda', 'models': {'classifier': True, 'detector': True, 'llm': True, 'retrieval': True}}
Streamlit: http://localhost:8501
↓ https://github.com/cloudflare/cloudflared/releases/latest/download/cloud

## 8. Sau khi xong

1. Tải `artifacts/` và `data/gallery/` từ Drive về máy.
2. Chép vào repo, rồi:

```bash
git add artifacts data/gallery && git commit -m "artifacts do tren Colab T4" && git push
```

3. Điền số thật vào `MODEL_CARD.md`.

### Nếu gặp lỗi

| Hiện tượng | Cách xử lý |
|---|---|
| Tải model Hugging Face lỗi 429 | `from huggingface_hub import login; login()` bằng token miễn phí |
| `CUDA out of memory` | Restart runtime rồi chạy lại từ cell 3 với `!python scripts/build_artifacts.py retrieval rag` |
| `serve.py` báo thiếu mô hình | Xem `logs/api.log`, chạy lại stage tương ứng |
| Link tunnel không mở | Chạy lại `!python scripts/serve.py tunnel` — mỗi lần ra link mới |
| Colab báo hết quota GPU | Đổi sang CPU, chấp nhận số yếu hơn, và ghi rõ trong báo cáo là đo trên CPU |
